In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from h5py import File

import csiborgtools
import healpy as hp
from scipy.special import spherical_jn
from scipy.interpolate import interp1d
from scipy.integrate import quad

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
k, Pk = np.load("/mnt/users/rstiskalek/BayesianBulkFlows/data/pk_fiducial.npy").T
loglog_Pk = interp1d(np.log(k), np.log(Pk))

def Pk(k):
    return np.exp(loglog_Pk(np.log(k)))

def djn(n, x):
    return (n / x) * spherical_jn(n, x) - spherical_jn(n + 1, x)


def C_ell_theory_equal_radius(ell, r):
    dDdtau = 35.5
    
    norm = dDdtau**2 
    integrand = lambda k: norm * Pk(k) * djn(ell, k * r)**2
    return quad(integrand, 0.0001, 10, limit=1000, limlst=5)[0]


In [ ]:
with File("/mnt/extraspace/rstiskalek/BBF/Quijote_Cell/C_ell_fiducial.h5", 'r') as f:
    ell = f["ell"][:]
    radii = f["radii"][:]
    
    for n in range(50):
        if n == 0:
            C_ell = np.full((50, *f[f"nsim_00000/C_ell"].shape), np.nan)

        C_ell[n] = f[f"nsim_{str(n).zfill(5)}/C_ell"][...]

    # skymap = {n: f[f"nsim_00000/skymap_{n}"][...] for n in range(len(radii))}

In [ ]:


fig, axs = plt.subplots(len(radii), 1, figsize=(5, 4 * len(radii)), sharex=True)

for k in range(len(radii)):

    ylow, ymed, yhigh = np.percentile(C_ell[:, k], [16, 50, 84], axis=0)
    axs[k].plot(ell, ymed, label=rf"$R = {radii[k]} ~ [\mathrm{{Mpc}} / h]$")
    axs[k].fill_between(ell, ylow, yhigh, alpha=0.1)

    Cell_theory = [C_ell_theory_equal_radius(ell_, radii[k]) for ell_ in ell]
    axs[k].plot(ell, Cell_theory, color="black", label="Theory")

    axs[k].set_yscale("log")
    axs[k].legend()

    axs[k].set_ylabel(r"$C_{\ell}$")
    axs[k].set_xlabel(r"$\ell$")

axs[-1].set_xlim(0, ell.max())

fig.tight_layout()
fig.savefig("../../plots/C_ell_fiducial.png", dpi=450)
fig.show()

In [ ]:
hp.mollview(skymap[4], unit=r"$V_{\rm rad} ~ [\mathrm{km} / \mathrm{s}]$",  cmap="coolwarm")

In [ ]:
k = 4

Cell_theory = [C_ell_theory_equal_radius(ell_, radii[k]) for ell_ in ell]

plt.figure()
plt.title(f"R = {radii[k]}")


for n in range(50):
    plt.plot(ell, C_ell[n, k], alpha=0.1, color="black")


plt.plot(ell, C_ell[0, k], color="red", label="Simulated")
plt.plot(ell, Cell_theory, color="blue", label="Theory")

plt.yscale("log")
plt.show()
